# Análise exploratória — dataset de Ilager et al. (2023)

Fonte: https://zenodo.org/records/10069402 (CC BY 4.0).

Antes de rodar:
1. Baixe `dataset1_combined.csv` e `dataset2_combined.csv` e coloque em `dados/brutos/` na raiz do repositório (o notebook deve ser aberto a partir da pasta `notebooks/`).
2. Instale as dependências do projeto com o grupo do notebook: `uv sync --group notebook`.
3. Rode a seção 1, confira as colunas impressas e ajuste o dicionário `COLUNAS` na seção 2.

## 1. Carregamento e inspeção

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

PASTA_DADOS = Path("..") / "dados" / "brutos"
PASTA_SAIDA = Path("saida")
PASTA_SAIDA.mkdir(exist_ok=True)

ARQUIVOS = {
    "D1": PASTA_DADOS / "dataset1_combined.csv",
    "D2": PASTA_DADOS / "dataset2_combined.csv",
}
INTERVALO = pd.Timedelta(minutes=10)

In [ ]:
amostra = pd.read_csv(ARQUIVOS["D1"], nrows=5)
print(list(amostra.columns))
amostra

## 2. Mapeamento das colunas

Os nomes abaixo são palpites. Ajuste o lado direito para os nomes reais impressos acima. Deixe `None` para colunas que não existirem.

In [ ]:
def adivinhar(colunas, chaves):
    for col in colunas:
        nome = col.lower()
        if any(k in nome for k in chaves):
            return col
    return None

cols = list(amostra.columns)
COLUNAS = {
    "ts": adivinhar(cols, ["time", "date", "ts"]),
    "host": adivinhar(cols, ["host", "node", "machine", "pm", "server"]),
    "potencia": adivinhar(cols, ["power", "watt", "energy"]),
    "cpu": adivinhar(cols, ["cpu_util", "cpu_usage", "cpu"]),
    "memoria": adivinhar(cols, ["mem", "ram"]),
    "vms": adivinhar(cols, ["vm"]),
}
COLUNAS

## 3. Leitura completa, padronização e conversão para Parquet

Na primeira execução o CSV é convertido para Parquet em `saida/`, e as próximas leituras ficam bem mais rápidas. Se mudar o `COLUNAS` depois disso, apague os arquivos `.parquet` da pasta `saida/` e rode de novo.

In [ ]:
def carregar(nome):
    destino = PASTA_SAIDA / f"{nome}.parquet"
    if destino.exists():
        return pd.read_parquet(destino)
    bruto = pd.read_csv(ARQUIVOS[nome], low_memory=False)
    mapa = {orig: novo for novo, orig in COLUNAS.items() if orig}
    df = bruto.rename(columns=mapa)
    ts = df["ts"]
    if pd.api.types.is_numeric_dtype(ts):
        unidade = "ms" if ts.max() > 1e11 else "s"
        df["ts"] = pd.to_datetime(ts, unit=unidade, utc=True)
    else:
        df["ts"] = pd.to_datetime(ts, utc=True, errors="coerce")
    df["host"] = df["host"].astype(str)
    df = df.sort_values(["host", "ts"]).reset_index(drop=True)
    df.to_parquet(destino, index=False)
    return df

dados = {nome: carregar(nome) for nome in ARQUIVOS if ARQUIVOS[nome].exists()}
for nome, df in dados.items():
    print(nome, df.shape, df["ts"].min(), "→", df["ts"].max(), "| servidores:", df["host"].nunique())

## 4. Qualidade dos dados

In [ ]:
def qualidade(df):
    linhas = len(df)
    dup = df.duplicated(["host", "ts"]).sum()
    ts_invalido = df["ts"].isna().sum()
    pot = df["potencia"]
    pot_invalida = (pot <= 0).sum()
    cpu_fora = ((df["cpu"] < 0) | (df["cpu"] > 100)).sum() if "cpu" in df else np.nan
    resumo = pd.Series({
        "linhas": linhas,
        "duplicados (host, ts)": dup,
        "timestamps inválidos": ts_invalido,
        "potência <= 0": pot_invalida,
        "CPU fora de 0–100": cpu_fora,
    })
    ausentes = (df.isna().mean() * 100).round(2).rename("% ausentes")
    return resumo, ausentes

for nome, df in dados.items():
    resumo, ausentes = qualidade(df)
    print(f"=== {nome} ===")
    display(resumo.to_frame("valor"))
    display(ausentes[ausentes > 0].to_frame() if (ausentes > 0).any() else "sem valores ausentes")

In [ ]:
def cobertura(df):
    inicio, fim = df["ts"].min(), df["ts"].max()
    esperado = int((fim - inicio) / INTERVALO) + 1
    por_host = df.drop_duplicates(["host", "ts"]).groupby("host").agg(
        leituras=("ts", "size"),
        primeira=("ts", "min"),
        ultima=("ts", "max"),
    )
    por_host["cobertura_%"] = (por_host["leituras"] / esperado * 100).round(1)
    passos = df.drop_duplicates(["host", "ts"]).groupby("host")["ts"].diff().dropna()
    return por_host.sort_values("cobertura_%"), passos.value_counts().head(10)

for nome, df in dados.items():
    por_host, passos = cobertura(df)
    print(f"=== {nome} === intervalos mais comuns entre leituras:")
    display(passos.to_frame("ocorrências"))
    display(por_host.head(10))
    por_host["cobertura_%"].plot.hist(bins=30, title=f"{nome} — cobertura por servidor (%)")
    plt.show()

## 5. Estatística descritiva

In [ ]:
NUMERICAS = [c for c in ["potencia", "cpu", "memoria", "vms"] if c in next(iter(dados.values()))]
for nome, df in dados.items():
    print(f"=== {nome} ===")
    display(df[NUMERICAS].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T.round(2))

## 6. Comportamento temporal do consumo agregado

In [ ]:
def agregado(df):
    limpo = df.drop_duplicates(["host", "ts"])
    serie = limpo.groupby("ts")["potencia"].sum().asfreq("10min")
    return serie

for nome, df in dados.items():
    serie = agregado(df)
    fig, ax = plt.subplots()
    (serie / 1000).plot(ax=ax, lw=0.6)
    ax.set_title(f"{nome} — potência total (kW), a cada 10 min")
    ax.set_ylabel("kW")
    plt.show()

    diario = (serie * (10 / 60) / 1000).resample("D").sum(min_count=1)
    fig, ax = plt.subplots()
    diario.plot(ax=ax, marker=".")
    ax.set_title(f"{nome} — energia diária (kWh)")
    ax.set_ylabel("kWh")
    plt.show()

In [ ]:
for nome, df in dados.items():
    serie = agregado(df).to_frame("potencia").dropna()
    serie["hora"] = serie.index.hour
    serie["dia_semana"] = serie.index.dayofweek
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    (serie.groupby("hora")["potencia"].mean() / 1000).plot(ax=axes[0], marker="o")
    axes[0].set_title(f"{nome} — perfil médio por hora (kW)")
    (serie.groupby("dia_semana")["potencia"].mean() / 1000).plot.bar(ax=axes[1])
    axes[1].set_title(f"{nome} — média por dia da semana (0 = segunda)")
    plt.tight_layout()
    plt.show()

## 7. Relação entre carga e potência

In [ ]:
for nome, df in dados.items():
    amostra_plot = df.dropna(subset=["cpu", "potencia"]).sample(min(200_000, len(df)), random_state=42)
    fig, ax = plt.subplots(figsize=(7, 5))
    hb = ax.hexbin(amostra_plot["cpu"], amostra_plot["potencia"], gridsize=60, bins="log", cmap="viridis")
    fig.colorbar(hb, label="log(contagem)")
    ax.set_xlabel("Utilização de CPU (%)")
    ax.set_ylabel("Potência")
    ax.set_title(f"{nome} — potência × CPU")
    plt.show()

In [ ]:
for nome, df in dados.items():
    corr = df[NUMERICAS + [c for c in df.select_dtypes("number").columns if c not in NUMERICAS]].corr()["potencia"]
    print(f"=== {nome} — correlação de Pearson com a potência ===")
    display(corr.drop("potencia").sort_values(ascending=False).round(3).to_frame())

## 8. Ociosidade

Potência ociosa de referência: percentil 5 da potência quando a CPU está abaixo de 5%.
Energia em ociosidade: energia consumida em intervalos com CPU abaixo do limiar.

In [ ]:
LIMIARES = [5, 10, 15]

def ociosidade(df):
    limpo = df.drop_duplicates(["host", "ts"]).dropna(subset=["cpu", "potencia"])
    energia = limpo["potencia"] * (10 / 60) / 1000
    linhas = {}
    for lim in LIMIARES:
        ocioso = limpo["cpu"] < lim
        linhas[f"CPU < {lim}%"] = {
            "% do tempo": round(ocioso.mean() * 100, 1),
            "energia (kWh)": round(energia[ocioso].sum(), 1),
            "% da energia": round(energia[ocioso].sum() / energia.sum() * 100, 1),
        }
    por_host = limpo.groupby("host").apply(
        lambda g: pd.Series({
            "p_idle": g.loc[g["cpu"] < 5, "potencia"].quantile(0.05),
            "p_max": g["potencia"].quantile(0.99),
            "cpu_media": g["cpu"].mean(),
        }),
        include_groups=False,
    )
    por_host["idle_sobre_max_%"] = (por_host["p_idle"] / por_host["p_max"] * 100).round(1)
    return pd.DataFrame(linhas).T, por_host

for nome, df in dados.items():
    tabela, por_host = ociosidade(df)
    print(f"=== {nome} ===")
    display(tabela)
    display(por_host.describe().round(1))

## 9. Distribuição da potência por servidor

In [ ]:
for nome, df in dados.items():
    medias = df.groupby("host")["potencia"].median().sort_values()
    escolhidos = list(medias.index[:5]) + list(medias.index[len(medias) // 2 - 2: len(medias) // 2 + 3]) + list(medias.index[-5:])
    sub = df[df["host"].isin(escolhidos)]
    fig, ax = plt.subplots(figsize=(14, 4))
    sub.boxplot(column="potencia", by="host", ax=ax, rot=90, showfliers=False)
    ax.set_title(f"{nome} — potência por servidor (menores, medianos e maiores)")
    plt.suptitle("")
    plt.show()

## 10. Baseline rápido de previsão (ingênuo sazonal)

Previsão do consumo agregado repetindo o valor do mesmo horário do dia anterior (144 intervalos de 10 min). Serve como referência mínima que os modelos de machine learning precisam superar.

> **Nota:** a divisão 80/20 desta seção é só um baseline rápido exploratório. A divisão oficial do projeto é cronológica 60/20/20 (treino/validação/teste), implementada na Fase 1.

In [ ]:
for nome, df in dados.items():
    serie = agregado(df).dropna()
    corte = int(len(serie) * 0.8)
    teste = serie.iloc[corte:]
    prev_sazonal = serie.shift(144).iloc[corte:]
    prev_ultimo = serie.shift(6).iloc[corte:]
    validos = teste.notna() & prev_sazonal.notna() & prev_ultimo.notna()
    mae_sazonal = (teste[validos] - prev_sazonal[validos]).abs().mean()
    mae_ultimo = (teste[validos] - prev_ultimo[validos]).abs().mean()
    print(f"{nome}: MAE ingênuo sazonal (24 h) = {mae_sazonal:,.1f} | MAE último valor (1 h antes) = {mae_ultimo:,.1f}")

## 11. Exportação dos resultados

In [ ]:
for nome, df in dados.items():
    resumo, ausentes = qualidade(df)
    por_host, _ = cobertura(df)
    tabela, idle = ociosidade(df)
    with pd.ExcelWriter(PASTA_SAIDA / f"resumo_{nome}.xlsx") as w:
        resumo.to_frame("valor").to_excel(w, sheet_name="qualidade")
        por_host.assign(primeira=por_host["primeira"].dt.tz_localize(None), ultima=por_host["ultima"].dt.tz_localize(None)).to_excel(w, sheet_name="cobertura")
        df[NUMERICAS].describe().T.to_excel(w, sheet_name="descritiva")
        tabela.to_excel(w, sheet_name="ociosidade")
        idle.to_excel(w, sheet_name="idle_por_servidor")
print("Arquivos salvos em", PASTA_SAIDA.resolve())